In [4]:
import numpy as np
from sklearn.utils.extmath import softmax
from matplotlib import pyplot as plt
import sklearn as skl
import re
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from sklearn.datasets import fetch_openml
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

In [5]:
# Dataset Preprocessing
#import data and class
#use very convenient sklearn functions to transform into polynomial
studentproddist = pd.read_csv("student_productivity_distraction_dataset_20000.csv")
df = pd.DataFrame(studentproddist)
oh = pd.get_dummies(df['gender'])
df = df.drop('gender',axis=1)
df = df.join(oh)
np.random.seed(1)
N=df.size
df = df.drop(columns=["student_id", "social_media_hours", "youtube_hours", "gaming_hours"])
y = df["phone_usage_hours"]
X = df.drop(columns=["phone_usage_hours"]) #add any other columns you don't want
print(X.shape)
sigma = 0.3
noise = np.random.normal(0,sigma,X.shape)
X1 = noise + X
noise = np.random.normal(0,sigma,y.shape)
y1 = noise + y
deg_list=[2,3,4]
polydict = {}
xtraind={}
xtestd={}
ytraind={}
ytestd={}
for i in deg_list:
    poly=PolynomialFeatures(degree=i,include_bias=False)
    polyf=poly.fit_transform(X)
    #polydict[i] = polyf
    xtraind[i], xtestd[i], ytraind[i], ytestd[i] = train_test_split(polyf, y, test_size=0.2)
    poly=PolynomialFeatures(degree=i,include_bias=False)
    polyf=poly.fit_transform(X1)
    #polydict[i] = polyf
    xtraind[-1 * i], xtestd[-1 * i], ytraind[-i], ytestd[-i] = train_test_split(polyf, y1, test_size=0.2)
df

(20000, 15)


,age,study_hours_per_day,sleep_hours,phone_usage_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,Female,Male,Other
0,23,4.35,3.63,3.38,6,347,111,2,57.21,10,57,81.87,33.78,True,False,False
1,20,6.14,6.58,5.48,13,403,28,10,91.27,10,49,60.90,48.99,False,True,False
2,29,4.98,3.26,4.83,1,419,102,8,63.14,2,38,86.22,36.60,True,False,False
3,27,3.19,4.58,10.06,9,178,28,18,40.51,6,50,71.77,19.87,True,False,False
4,24,7.67,6.21,3.02,8,436,105,7,45.53,6,41,90.13,52.90,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,26,4.81,7.83,6.86,6,451,0,11,63.69,3,89,82.13,60.43,True,False,False
19996,22,0.83,4.49,6.76,1,375,13,4,74.06,10,56,87.12,17.84,False,False,True
19997,25,9.36,4.33,2.94,4,413,1,11,73.18,9,69,81.58,65.56,False,True,False
19998,22,0.74,4.44,3.15,7,319,1,6,77.02,5,53,53.04,28.06,True,False,False


## Polynomial Regression

In [89]:
from sklearn.linear_model import LinearRegression
for i in deg_list: #standard, no noise
    polyregmodel=LinearRegression()
    polyregmodel.fit(xtraind[i],ytraind[i])
    ppred = polyregmodel.predict(xtestd[i])
    pregrms = np.sqrt(skl.metrics.mean_squared_error(ytestd[i],ppred))
    print('loss for degree ', i)
    print(pregrms)

loss for degree  2
0.0017817246481202904
loss for degree  3
0.0017995430743786732
loss for degree  4
0.012612176605102226


In [14]:
from sklearn.linear_model import LinearRegression
for i in deg_list: #standard, no noise
    polyregmodel=LinearRegression()
    polyregmodel.fit(xtraind[-i],ytraind[-i])
    ppred = polyregmodel.predict(xtestd[-i])
    pregrms = np.sqrt(skl.metrics.mean_squared_error(ytestd[-i],ppred))
    trainpred = polyregmodel.predict(xtraind[-i])
    trainloss = np.sqrt(skl.metrics.mean_squared_error(ytraind[-i],trainpred))
    print('training and test scores,k=5,with noise for degree ', i)
    print(cross_val_score(polyregmodel, xtraind[-i], ytraind[-i],cv=5))
    print(cross_val_score(polyregmodel, xtestd[-i], ytestd[-i],cv=5))
    print('training and validation loss: ')
    print(pregrms)
    print(trainloss)
    print('\n')

training and test scores,k=5,with noise for degree  2
[0.91402145 0.91015925 0.91071081 0.9131029  0.91294186]
[0.91461053 0.90900499 0.90063235 0.90641859 0.90235013]
training and validation loss: 
0.9974993162869699
0.9763328343669301


training and test scores,k=5,with noise for degree  3
[0.91681043 0.91481551 0.9127847  0.91149624 0.9145111 ]
[0.89482478 0.88723408 0.90103057 0.89287103 0.89144751]
training and validation loss: 
0.9569306552354768
0.919093785937181


training and test scores,k=5,with noise for degree  4
[0.87709974 0.87660154 0.87550041 0.8760956  0.88329284]
[-6.46321077 -6.42778572 -6.23562045 -6.21912925 -6.04191731]
training and validation loss: 
1.1068079358598342
0.8256203979613812




loss for degree  2
4.630297743430828
loss for degree  3
4.55963467394363
loss for degree  4
1.1025671263657786


In [ ]:

for i in deg_list: #Bayesian
    clf = skl.linear_model.BayesianRidge()
    baypolyreg = clf.fit(xtraind[i],ytraind[i])
    baypred = baypolyreg.predict(xtestd[i])
    pregrms = np.sqrt(skl.metrics.mean_squared_error(ytestd[i],ppred))
    print('loss for degree ', i)
    print(pregrms)

In [7]:
print("Bayesian Polynomials")
for i in deg_list: #Bayesian
    clf = skl.linear_model.BayesianRidge()
    baypolyreg = clf.fit(xtraind[-i],ytraind[-i])
    baypred = baypolyreg.predict(xtestd[-i])
    pregrms = np.sqrt(skl.metrics.mean_squared_error(ytestd[-i],baypred))
    baytrain = baypolyreg.predict(xtraind[-i])
    trainloss = np.sqrt(skl.metrics.mean_squared_error(ytraind[-i],baytrain))
    print('training and test cross validation scores with noise for degree ', i)
    print(cross_val_score(baypolyreg,xtraind[-i],ytraind[-i],cv=5))
    print(cross_val_score(baypolyreg,xtestd[-i],ytestd[-i],cv=5))
    print('training and test loss with noise for degree ', i)
    print(trainloss)
    print(pregrms)
    print('\n')
    

Bayesian Polynomials
training and test cross validation scores with noise for degree  2
[0.90826845 0.90957969 0.91476275 0.90556638 0.9115064 ]
[0.90783405 0.90653807 0.90003789 0.90212821 0.89898233]
training and test loss with noise for degree  2
0.9889413255922218
0.9759763766831662


training and test cross validation scores with noise for degree  3
[0.90518908 0.90736629 0.90209055 0.90933801 0.90182763]
[0.87316321 0.88505765 0.8883302  0.87703304 0.88558241]
training and test loss with noise for degree  3
0.976707762991997
1.0049181106299776


training and test cross validation scores with noise for degree  4
[0.89725399 0.89875036 0.89294755 0.89488308 0.90005153]
[0.79907274 0.78147854 0.79399456 0.79301393 0.81633378]
training and test loss with noise for degree  4
0.9101859349608342
1.0341270304699748




In [ ]:
#one layer example, quite poor compared to the above.
from sklearn.neural_network import MLPRegressor

solver = MLPRegressor(hidden_layer_sizes=(15,),batch_size=1000,learning_rate_init=1e-3,
                      max_iter = 1000,activation='logistic',solver='adam')
for i in deg_list:
    solver.fit(X=xtraind[-i], y=ytraind[-i])
    print(cross_val_score(solver, xtestd[-i],ytestd[-i],cv=5))



[-0.00417836 -0.0025676  -0.00106894 -0.00449532 -0.12144807]


c:\Users\chaos\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


[-0.00663823 -0.00960667 -0.04272889 -0.00789083 -0.29802293]
[-1.02355544e-03 -2.50277965e-07 -1.04721216e-01 -6.12264742e-03
 -1.97445010e-05]
